# Lesson 01 - Introduction to AI Agents

Welcome to the first lesson in the **AI Agents for Beginners** course!

An **AI agent** is a program that uses a large language model (LLM) as its reasoning engine and can take *actions* in the real world — calling APIs, querying databases, or running code — to accomplish a goal on behalf of a user.

In this notebook you will build your first agent: a **Travel Agent** that recommends vacation destinations. Along the way you will learn how to:

1. Connect to Azure AI Foundry Agent Service using the **Microsoft Agent Framework**.
2. Give the agent a **tool** — a plain Python function it can call.
3. Run the agent and inspect its response.
4. Stream the agent's response token-by-token.

## Setup

Before running this notebook, make sure you have:

1. **An Azure AI Foundry project** with a deployed chat model (e.g. `gpt-4o-mini`).
2. **Logged in with the Azure CLI** — run `az login` in your terminal.
3. **Set the required environment variables:**
   - `AZURE_AI_PROJECT_ENDPOINT` — your Azure AI Foundry project endpoint.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — the name of your deployed model.

The cell below installs the Python packages you need.

In [1]:
%pip install agent-framework azure-ai-projects azure-identity -q


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Creating Your First Agent

An agent needs two things:

- **Instructions** that tell it *who it is* and *how to behave* (a system prompt).
- **Tools** — Python functions decorated with `@tool` that the agent can call to retrieve information or perform actions.

Below we define a simple tool that returns a list of popular vacation destinations. The agent will use this tool when a user asks for travel recommendations.

In [7]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
        "Ierapetra",
        "Agia Triada"
    ]

In [8]:
agent = await provider.create_agent(
    tools=[get_destinations],
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

Here are a few wonderful warm beach destinations from the list:  

1. **Bali, Indonesia** – Famed for its stunning beaches, lively culture, and tropical atmosphere, Bali offers a perfect beach paradise.  
2. **Ierapetra, Crete (Greece)** – Located on the southern coast of Crete, it boasts pristine beaches and Mediterranean charm.  
3. **Agia Triada, Crete (Greece)** – Another fantastic location in Crete, with warm weather and serene sandy beaches.  
4. **Rio de Janeiro, Brazil** – Home to the iconic Copacabana and Ipanema beaches, it combines vibrant beach life and a unique cultural experience.

Would you like help narrowing down or learning more about any of these destinations?


## Streaming Responses

For a more interactive experience you can **stream** the agent's response. Instead of waiting for the full reply, the agent yields text chunks as they are generated. This is especially useful in chat interfaces where you want to display output in real time.

In [10]:
async for chunk in agent.run(
    "Tell me about Bologna as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

Bologna, often referred to as the "Red City" due to its terracotta-colored buildings, is a charming and historic travel destination located in northern Italy. It’s the capital of the Emilia-Romagna region and is well-known for its culinary delights, medieval architecture, and vibrant cultural scene. Here's an overview of what you can expect when visiting Bologna:

### **Top Attractions**
1. **Piazza Maggiore and Piazza del Nettuno**: Located in the heart of the city, these neighboring squares are surrounded by historic buildings such as the Basilica di San Petronio and the medieval Palazzo d'Accursio. The Neptune Fountain (Fontana del Nettuno) is a highlight here.
   
2. **Le Due Torri (The Two Towers)**: The iconic Asinelli and Garisenda towers are symbols of Bologna. Visitors can climb the Asinelli Tower for breathtaking panoramic views of the city.

3. **Basilica di San Petronio**: One of the largest churches in the world, this Gothic basilica is renowned for its stunning interior a

## Summary

In this lesson you learned how to:

- **Create a provider** that connects to Azure AI Foundry Agent Service via `AzureAIProjectAgentProvider`.
- **Define a tool** using the `@tool` decorator so the agent can call your Python functions.
- **Run the agent** with a user message and print its response.
- **Stream responses** for real-time output.

In the next lesson we will explore agentic frameworks in more depth and learn how to give agents more powerful tools and multi-step reasoning capabilities.